# Demo NLP: Phân loại cảm xúc tiếng Việt bằng LSTM

Notebook này minh họa một pipeline NLP cơ bản cho bài toán **phân loại cảm xúc review tiếng Việt**.

Mục tiêu của demo:

- Chuẩn bị một tập dữ liệu review nhỏ.
- Tiền xử lý văn bản bằng Tokenizer và Padding.
- Xây dựng mô hình LSTM.
- Huấn luyện với train/validation set.
- Đánh giá trên test set.
- Thử dự đoán cảm xúc của câu mới.

> Lưu ý: Dataset trong demo nhỏ, nên mục tiêu chính là hiểu quy trình NLP chứ không phải đạt độ chính xác rất cao.


In [ ]:
# =========================
# 0. CÀI THƯ VIỆN NẾU MÁY CHƯA CÓ
# =========================
# Cell này giúp notebook chạy được trên môi trường mới.
# Trên Google Colab thường đã có sẵn hầu hết thư viện.

import importlib.util
import subprocess
import sys

required_packages = {
    "numpy": "numpy",
    "pandas": "pandas",
    "matplotlib": "matplotlib",
    "sklearn": "scikit-learn",
    "tensorflow": "tensorflow"
}

for import_name, pip_name in required_packages.items():
    if importlib.util.find_spec(import_name) is None:
        print(f"Đang cài {pip_name} ...")
        subprocess.check_call([sys.executable, "-m", "pip", "install", "-q", pip_name])

print("Kiểm tra thư viện xong.")


In [ ]:
# =========================
# 1. IMPORT THƯ VIỆN
# =========================

import os
import random
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt

from sklearn.model_selection import train_test_split
from sklearn.metrics import classification_report, confusion_matrix, ConfusionMatrixDisplay

import tensorflow as tf
from tensorflow.keras.preprocessing.text import Tokenizer
from tensorflow.keras.preprocessing.sequence import pad_sequences
from tensorflow.keras.models import Sequential
from tensorflow.keras.layers import Embedding, SpatialDropout1D, LSTM, Dense, Dropout
from tensorflow.keras.callbacks import EarlyStopping
from tensorflow.keras.optimizers import Adam

# Cố định seed để kết quả ổn định hơn qua nhiều lần chạy
SEED = 42
random.seed(SEED)
np.random.seed(SEED)
tf.random.set_seed(SEED)

print("TensorFlow version:", tf.__version__)


## 2. Tạo dữ liệu mẫu

Trong demo này, ta tạo một dataset nhỏ gồm các câu review tiếng Việt.

Quy ước nhãn:

- `1`: tích cực
- `0`: tiêu cực

Dataset được viết trực tiếp trong notebook để dễ chạy, không cần tải file bên ngoài.


In [ ]:
# =========================
# 2. DATASET MẪU
# =========================

positive_reviews = [
    "phim rất hay và cảm động",
    "diễn viên diễn xuất tự nhiên",
    "cốt truyện hấp dẫn từ đầu đến cuối",
    "âm nhạc trong phim rất tuyệt vời",
    "mình rất thích bộ phim này",
    "nội dung ý nghĩa và dễ xem",
    "hình ảnh đẹp màu sắc dễ chịu",
    "phim làm mình cảm thấy vui",
    "kịch bản tốt và có chiều sâu",
    "đây là một trải nghiệm xem phim đáng nhớ",
    "nhân vật chính được xây dựng rất tốt",
    "phim có nhiều cảnh cảm xúc",
    "mình sẽ giới thiệu phim này cho bạn bè",
    "chất lượng phim vượt ngoài mong đợi",
    "bộ phim đem lại cảm giác tích cực",
    "lời thoại duyên dáng và gần gũi",
    "mạch phim ổn không bị lê thê",
    "đạo diễn xử lý câu chuyện rất khéo",
    "phần kết khiến mình hài lòng",
    "phim đáng xem vào cuối tuần",
    "tôi thích cách phim truyền tải thông điệp",
    "mọi thứ trong phim khá trọn vẹn",
    "diễn xuất của dàn cast rất thuyết phục",
    "phim nhẹ nhàng nhưng vẫn cuốn hút",
    "nhiều phân đoạn làm tôi xúc động"
]

negative_reviews = [
    "phim rất chán và dài dòng",
    "diễn viên diễn xuất gượng gạo",
    "cốt truyện rời rạc khó hiểu",
    "âm thanh tệ và gây khó chịu",
    "mình không thích bộ phim này",
    "nội dung nhạt và thiếu điểm nhấn",
    "hình ảnh xấu màu sắc lộn xộn",
    "phim làm mình cảm thấy thất vọng",
    "kịch bản yếu và thiếu logic",
    "đây là một trải nghiệm xem phim tệ",
    "nhân vật chính xây dựng quá hời hợt",
    "phim có nhiều cảnh thừa thãi",
    "mình sẽ không giới thiệu phim này",
    "chất lượng phim thấp hơn mong đợi",
    "bộ phim đem lại cảm giác mệt mỏi",
    "lời thoại cứng và thiếu tự nhiên",
    "mạch phim chậm và khá buồn ngủ",
    "đạo diễn xử lý câu chuyện chưa tốt",
    "phần kết khiến mình khó chịu",
    "phim không đáng xem vào cuối tuần",
    "tôi không thích cách phim truyền tải thông điệp",
    "mọi thứ trong phim khá lộn xộn",
    "diễn xuất của dàn cast chưa thuyết phục",
    "phim nặng nề nhưng không cuốn hút",
    "nhiều phân đoạn làm tôi mất kiên nhẫn"
]

texts = positive_reviews + negative_reviews
labels = [1] * len(positive_reviews) + [0] * len(negative_reviews)

df = pd.DataFrame({
    "text": texts,
    "label": labels
})

# Trộn dữ liệu để câu tích cực và tiêu cực không đứng thành từng cụm
df = df.sample(frac=1, random_state=SEED).reset_index(drop=True)

df.head(10)


In [ ]:
# Xem số lượng mẫu theo từng nhãn
print(df["label"].value_counts())

label_names = {0: "Tiêu cực", 1: "Tích cực"}
df["sentiment"] = df["label"].map(label_names)
df.head()


## 3. Chia train, validation và test

Ta chia dữ liệu thành 3 phần:

- **Train set**: dùng để mô hình học.
- **Validation set**: dùng để theo dõi quá trình học và hỗ trợ EarlyStopping.
- **Test set**: chỉ dùng để đánh giá cuối cùng.

Việc giữ riêng test set giúp kết quả đánh giá khách quan hơn.


In [ ]:
# =========================
# 3. CHIA DỮ LIỆU
# =========================

X = df["text"].values
y = df["label"].values

# Tách test set trước
X_train_full, X_test, y_train_full, y_test = train_test_split(
    X, y,
    test_size=0.20,
    random_state=SEED,
    stratify=y
)

# Từ train_full tách tiếp validation set
X_train, X_val, y_train, y_val = train_test_split(
    X_train_full, y_train_full,
    test_size=0.25,
    random_state=SEED,
    stratify=y_train_full
)

print("Số mẫu train:", len(X_train))
print("Số mẫu validation:", len(X_val))
print("Số mẫu test:", len(X_test))


## 4. Tokenization và Padding

Mô hình không xử lý trực tiếp được chữ, nên ta cần chuyển câu thành dãy số.

Các bước chính:

1. `Tokenizer` học bộ từ vựng từ tập train.
2. Mỗi câu được biến thành một chuỗi token id.
3. Các chuỗi được padding về cùng độ dài để đưa vào mô hình.


In [ ]:
# =========================
# 4. TOKENIZATION + PADDING
# =========================

MAX_WORDS = 2000
MAX_LEN = 20
OOV_TOKEN = "<OOV>"

tokenizer = Tokenizer(num_words=MAX_WORDS, oov_token=OOV_TOKEN)
tokenizer.fit_on_texts(X_train)

X_train_seq = tokenizer.texts_to_sequences(X_train)
X_val_seq = tokenizer.texts_to_sequences(X_val)
X_test_seq = tokenizer.texts_to_sequences(X_test)

X_train_pad = pad_sequences(X_train_seq, maxlen=MAX_LEN, padding="post", truncating="post")
X_val_pad = pad_sequences(X_val_seq, maxlen=MAX_LEN, padding="post", truncating="post")
X_test_pad = pad_sequences(X_test_seq, maxlen=MAX_LEN, padding="post", truncating="post")

vocab_size = min(MAX_WORDS, len(tokenizer.word_index) + 1)

print("Kích thước từ vựng:", vocab_size)
print("Ví dụ câu gốc:", X_train[0])
print("Sau khi tokenize:", X_train_seq[0])
print("Sau khi padding:", X_train_pad[0])


## 5. Xây dựng mô hình LSTM

Kiến trúc mô hình gồm:

- `Embedding`: biến token id thành vector.
- `SpatialDropout1D`: giảm overfitting ở tầng embedding.
- `LSTM`: học thông tin theo thứ tự từ trong câu.
- `Dense`: phân loại dựa trên đặc trưng đã học.
- `Sigmoid`: xuất xác suất câu thuộc lớp tích cực.


In [ ]:
# =========================
# 5. XÂY DỰNG MÔ HÌNH
# =========================

EMBEDDING_DIM = 64

model = Sequential([
    Embedding(input_dim=vocab_size, output_dim=EMBEDDING_DIM, mask_zero=True),
    SpatialDropout1D(0.10),
    LSTM(48, dropout=0.10),
    Dense(32, activation="relu"),
    Dropout(0.20),
    Dense(1, activation="sigmoid")
])

model.compile(
    loss="binary_crossentropy",
    optimizer=Adam(learning_rate=0.001),
    metrics=["accuracy"]
)

model.summary()


## 6. Huấn luyện mô hình

Vì dataset nhỏ nên mô hình dễ bị học thuộc. Do đó ta dùng `EarlyStopping` để dừng sớm khi validation loss không còn cải thiện.


In [ ]:
# =========================
# 6. TRAIN MODEL
# =========================

early_stop = EarlyStopping(
    monitor="val_loss",
    patience=8,
    restore_best_weights=True
)

history = model.fit(
    X_train_pad, y_train,
    validation_data=(X_val_pad, y_val),
    epochs=50,
    batch_size=4,
    callbacks=[early_stop],
    verbose=1
)


## 7. Vẽ biểu đồ Loss và Accuracy

Biểu đồ giúp quan sát mô hình học như thế nào qua từng epoch.


In [ ]:
# =========================
# 7. VẼ BIỂU ĐỒ TRAINING
# =========================

history_df = pd.DataFrame(history.history)
history_df.head()

plt.figure(figsize=(8, 5))
plt.plot(history_df["loss"], label="Train loss")
plt.plot(history_df["val_loss"], label="Validation loss")
plt.xlabel("Epoch")
plt.ylabel("Loss")
plt.title("Loss trong quá trình huấn luyện")
plt.legend()
plt.show()

plt.figure(figsize=(8, 5))
plt.plot(history_df["accuracy"], label="Train accuracy")
plt.plot(history_df["val_accuracy"], label="Validation accuracy")
plt.xlabel("Epoch")
plt.ylabel("Accuracy")
plt.title("Accuracy trong quá trình huấn luyện")
plt.legend()
plt.show()


## 8. Đánh giá trên test set

Sau khi train xong, ta dùng test set để kiểm tra mô hình trên dữ liệu chưa từng thấy.


In [ ]:
# =========================
# 8. EVALUATE
# =========================

test_loss, test_acc = model.evaluate(X_test_pad, y_test, verbose=0)

print(f"Test Loss: {test_loss:.4f}")
print(f"Test Accuracy: {test_acc:.4f}")

y_prob = model.predict(X_test_pad)
y_pred = (y_prob >= 0.5).astype(int).reshape(-1)

print("\nClassification Report:")
print(classification_report(
    y_test,
    y_pred,
    target_names=["Tiêu cực", "Tích cực"],
    zero_division=0
))

cm = confusion_matrix(y_test, y_pred)
disp = ConfusionMatrixDisplay(
    confusion_matrix=cm,
    display_labels=["Tiêu cực", "Tích cực"]
)
disp.plot()
plt.title("Confusion Matrix")
plt.show()


## 9. Thử dự đoán câu mới

Hàm dưới đây nhận một câu review tiếng Việt và trả về dự đoán của mô hình.


In [ ]:
# =========================
# 9. HÀM DỰ ĐOÁN CÂU MỚI
# =========================

def predict_sentiment(sentence):
    seq = tokenizer.texts_to_sequences([sentence])
    pad = pad_sequences(seq, maxlen=MAX_LEN, padding="post", truncating="post")
    prob_positive = float(model.predict(pad, verbose=0)[0][0])

    if prob_positive >= 0.5:
        label = "Tích cực"
        confidence = prob_positive
    else:
        label = "Tiêu cực"
        confidence = 1 - prob_positive

    return {
        "sentence": sentence,
        "predicted_label": label,
        "positive_probability": round(prob_positive, 4),
        "confidence": round(confidence, 4)
    }

test_sentences = [
    "phim hay và có nhiều cảm xúc",
    "nội dung quá chán và khó hiểu",
    "diễn viên diễn xuất rất tốt",
    "mình không muốn xem lại phim này",
    "phim ổn nhưng phần kết hơi thất vọng"
]

for s in test_sentences:
    print(predict_sentiment(s))


## 10. Kết luận

Qua demo này, ta đã xây dựng được một pipeline NLP hoàn chỉnh:

```text
Dữ liệu văn bản
→ Tokenization
→ Padding
→ Chia train/validation/test
→ Xây dựng mô hình LSTM
→ Huấn luyện
→ Đánh giá
→ Dự đoán câu mới
```

Một số hạn chế của demo:

- Dataset còn nhỏ nên mô hình chưa đủ mạnh để dùng trong thực tế.
- Chưa xử lý tách từ tiếng Việt chuyên sâu.
- Chưa dùng các mô hình pretrained như PhoBERT.

Nếu phát triển tiếp, có thể mở rộng dataset, dùng word embedding tiếng Việt hoặc fine-tune mô hình Transformer cho kết quả tốt hơn.


## Gợi ý thuyết trình ngắn

Trong demo này, nhóm em xây dựng một mô hình LSTM để phân loại cảm xúc review tiếng Việt. Đầu tiên, dữ liệu văn bản được chuyển thành các token số bằng Tokenizer, sau đó padding để các câu có cùng độ dài. Tiếp theo, mô hình dùng lớp Embedding để học biểu diễn từ và lớp LSTM để học thông tin theo thứ tự trong câu. Cuối cùng, mô hình dự đoán xác suất câu thuộc cảm xúc tích cực hay tiêu cực bằng hàm sigmoid. Vì dataset còn nhỏ nên mục tiêu chính của demo là minh họa quy trình xử lý NLP end-to-end.
